# VALENCE v0.8.0 - One-Click Colab Release

This notebook contains the complete public VALENCE v0.8.0 source snapshot.
It installs the package, runs all 75 tests, exercises the minimal and mainnet
Ethereum-calibrated profiles, and runs the 30-seed cross-model validation.

Outputs persist under `My Drive/valence/results/public-v0.8.0`.
Choose **Runtime -> Run all**.


In [ ]:
# STEP 1 - Mount Drive and define persistent paths.
from google.colab import drive
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile, base64
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/valence")
DRIVE_SOURCE = DRIVE_ROOT / "source-v0.8.0"
DRIVE_RESULTS = DRIVE_ROOT / "results" / "public-v0.8.0"
DRIVE_TESTS = DRIVE_ROOT / "test-results"
WORK_ROOT = Path("/content/valence_v0_8_workspace")
for p in (DRIVE_ROOT, DRIVE_RESULTS, DRIVE_TESTS): p.mkdir(parents=True, exist_ok=True)
print(DRIVE_ROOT)


In [ ]:
# STEP 2 - Reconstruct the source repository.
ARCHIVE_B64 = """
...
""".replace("\n", "").strip()
if WORK_ROOT.exists(): shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
archive_path = WORK_ROOT / "valence-public-v0.8.0.zip"
archive_path.write_bytes(base64.b64decode(ARCHIVE_B64))
with zipfile.ZipFile(archive_path) as z: z.extractall(WORK_ROOT)
RUNTIME_SOURCE = WORK_ROOT / "valence-public-v0.8.0"
if DRIVE_SOURCE.exists(): shutil.rmtree(DRIVE_SOURCE)
shutil.copytree(RUNTIME_SOURCE, DRIVE_SOURCE)
shutil.copy2(archive_path, DRIVE_ROOT / archive_path.name)
SOURCE = RUNTIME_SOURCE
print(f"Runtime source: {SOURCE}")
print(f"Persistent source: {DRIVE_SOURCE}")


In [ ]:
# RESUME GUARD
from pathlib import Path
import json, os, shutil, subprocess, sys
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/valence")
DRIVE_SOURCE = DRIVE_ROOT / "source-v0.8.0"
DRIVE_RESULTS = DRIVE_ROOT / "results" / "public-v0.8.0"
WORK_ROOT = Path("/content/valence_v0_8_workspace")
RUNTIME_SOURCE = WORK_ROOT / "valence-public-v0.8.0"
SOURCE = RUNTIME_SOURCE if (RUNTIME_SOURCE / "pyproject.toml").exists() else DRIVE_SOURCE
if not (SOURCE / "pyproject.toml").exists():
    raise RuntimeError("Run Steps 1 and 2 once to reconstruct VALENCE v0.8.0.")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
os.environ["PYTHONPATH"] = str(SOURCE / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
print(f"Source: {SOURCE}")
print(f"Results: {DRIVE_RESULTS}")

# STEP 3 - Install and run the 75-test release oracle.
subprocess.run([sys.executable,"-m","pip","install","--quiet","--no-build-isolation","-e",f"{SOURCE}[dev]"],check=True)
report = DRIVE_TESTS / "pytest-v0.8.0.txt"
run = subprocess.run([sys.executable,"-m","pytest","-q",str(SOURCE/"tests")],cwd=SOURCE,text=True,capture_output=True)
report.write_text(run.stdout + "\n" + run.stderr)
print(run.stdout)
if run.returncode != 0 or "75 passed" not in run.stdout:
    raise RuntimeError(f"Release tests failed; see {report}")


In [ ]:
# RESUME GUARD
from pathlib import Path
import json, os, shutil, subprocess, sys
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/valence")
DRIVE_SOURCE = DRIVE_ROOT / "source-v0.8.0"
DRIVE_RESULTS = DRIVE_ROOT / "results" / "public-v0.8.0"
WORK_ROOT = Path("/content/valence_v0_8_workspace")
RUNTIME_SOURCE = WORK_ROOT / "valence-public-v0.8.0"
SOURCE = RUNTIME_SOURCE if (RUNTIME_SOURCE / "pyproject.toml").exists() else DRIVE_SOURCE
if not (SOURCE / "pyproject.toml").exists():
    raise RuntimeError("Run Steps 1 and 2 once to reconstruct VALENCE v0.8.0.")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
os.environ["PYTHONPATH"] = str(SOURCE / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
print(f"Source: {SOURCE}")
print(f"Results: {DRIVE_RESULTS}")

# STEP 4 - Run minimal and mainnet profile smoke tests.
for preset in ("minimal", "mainnet"):
    config = SOURCE / "configs" / f"ethereum_{preset}_smoke.yaml"
    output = DRIVE_RESULTS / f"ethereum-{preset}-smoke"
    if output.exists(): shutil.rmtree(output)
    subprocess.run(["valence","run",str(config),"--output",str(output)],cwd=SOURCE,check=True)
    summary=json.loads((output/"summary.json").read_text())
    profile=summary["protocol_profile"]
    print(preset, profile)
    assert profile["ethereum_spec_release"] == "v1.6.1"
    assert profile["ethereum_fork"] == "fulu"
    assert profile["randao_mode"] == "deterministic_simulation_surrogate"
    assert profile["proposer_score_boost_applied"] is False


In [ ]:
# RESUME GUARD
from pathlib import Path
import json, os, shutil, subprocess, sys
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/valence")
DRIVE_SOURCE = DRIVE_ROOT / "source-v0.8.0"
DRIVE_RESULTS = DRIVE_ROOT / "results" / "public-v0.8.0"
WORK_ROOT = Path("/content/valence_v0_8_workspace")
RUNTIME_SOURCE = WORK_ROOT / "valence-public-v0.8.0"
SOURCE = RUNTIME_SOURCE if (RUNTIME_SOURCE / "pyproject.toml").exists() else DRIVE_SOURCE
if not (SOURCE / "pyproject.toml").exists():
    raise RuntimeError("Run Steps 1 and 2 once to reconstruct VALENCE v0.8.0.")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
os.environ["PYTHONPATH"] = str(SOURCE / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
print(f"Source: {SOURCE}")
print(f"Results: {DRIVE_RESULTS}")

# STEP 5 - Run or resume the 30-seed cross-model validation.
output = DRIVE_RESULTS / "ethereum-cross-model-30"
analysis = output / "analysis.json"
complete=False
if analysis.exists():
    data=json.loads(analysis.read_text())
    complete=data.get("design",{}).get("paired_seed_count") == 30
if not complete:
    subprocess.run([sys.executable,str(SOURCE/"scripts"/"run_ethereum_cross_model_experiment.py"),"--output",str(output)],cwd=SOURCE,check=True)
data=json.loads(analysis.read_text())
print(json.dumps(data["interaction_statistics"],indent=2))


In [ ]:
# RESUME GUARD
from pathlib import Path
import json, os, shutil, subprocess, sys
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/valence")
DRIVE_SOURCE = DRIVE_ROOT / "source-v0.8.0"
DRIVE_RESULTS = DRIVE_ROOT / "results" / "public-v0.8.0"
WORK_ROOT = Path("/content/valence_v0_8_workspace")
RUNTIME_SOURCE = WORK_ROOT / "valence-public-v0.8.0"
SOURCE = RUNTIME_SOURCE if (RUNTIME_SOURCE / "pyproject.toml").exists() else DRIVE_SOURCE
if not (SOURCE / "pyproject.toml").exists():
    raise RuntimeError("Run Steps 1 and 2 once to reconstruct VALENCE v0.8.0.")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
os.environ["PYTHONPATH"] = str(SOURCE / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
print(f"Source: {SOURCE}")
print(f"Results: {DRIVE_RESULTS}")

# STEP 6 - Show persistent outputs.
print("VALENCE v0.8.0 complete")
print(f"Source archive: {DRIVE_ROOT / 'valence-public-v0.8.0.zip'}")
print(f"Test report: {DRIVE_TESTS / 'pytest-v0.8.0.txt'}")
print(f"Results: {DRIVE_RESULTS}")
